# Why Subclassing API > Numpy made NN?

Subclassing allows total control over the training process, while the hard loops and blocks are being maintained by tensorflow. (Differentiation, layers, neurons, inputs) It is like what I have made using only numpy, but this once scales, I can add more layers, more activation functions and more kernel initializations more easily.

In [1]:
import numpy as np
import tensorflow as tf

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" # silence warnings

### Dataset

In [2]:
mnist = tf.keras.datasets.mnist
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

# 2. Flatten (28x28 -> 784) and normalize pixel values to [0.0, 1.0] (float32)
X_full = (X_train_raw.reshape(-1, 28 * 28) / 255.0).astype("float32")
X_test = (X_test_raw.reshape(-1, 28 * 28) / 255.0).astype("float32")

# 3. One-hot encode labels to match CategoricalCrossentropy (shape: (N, 10))
Y_full = tf.keras.utils.to_categorical(y_train_raw, num_classes=10).astype(
    "float32"
)
Y_test = tf.keras.utils.to_categorical(y_test_raw, num_classes=10).astype(
    "float32"
)

# 4. Create train (55,000) and validation (5,000) splits
X_val, X = X_full[:5000], X_full[5000:]
Y_val, Y = Y_full[:5000], Y_full[5000:]

# 5. Build tf.data Datasets
batch_size = 128

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X, Y))
    .shuffle(buffer_size=len(X))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [3]:
# 1. Initialization of layers
class CustomMLP(tf.keras.Model): # give the powers of tensorflow through inheritance
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(128, activation="relu", kernel_initializer="he_normal")
        self.dense2 = tf.keras.layers.Dense(128, activation="relu", kernel_initializer="he_normal")
        self.dense3 = tf.keras.layers.Dense(64, activation="silu", kernel_initializer="he_normal")
        self.output_layer = tf.keras.layers.Dense(10, activation="softmax")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        output = self.output_layer(x)
        return output

# 2. Hyperparameters
epochs = 100
batch_size = 128
patience = 9

model = CustomMLP()
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001, rho=0.95, epsilon=1e-8)
loss_fn = tf.keras.losses.CategoricalCrossentropy()

train_dataset = (tf.data.Dataset.from_tensor_slices((X, Y)).shuffle(len(X)).batch(batch_size))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(batch_size)

# 3. Early stopper tracker
best_val_loss = float("inf")
patience_counter = 0
best_weights = None
best_epoch = None

print("Starting Training...\n" + "-" * 45)

# 4. Epoch loop
for epoch in range(epochs):
    epoch_losses = []
    epoch_accs = []

    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            preds = model(x_batch)
            loss = loss_fn(y_batch, preds)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        # batch metrics
        acc = tf.reduce_mean(tf.cast(tf.argmax(preds, axis=1) == tf.argmax(y_batch, axis=1), tf.float32))
        epoch_losses.append(loss.numpy())
        epoch_accs.append(acc.numpy())

        epoch_losses.append(loss.numpy())
        epoch_accs.append(acc.numpy())

    # validation phase
    val_losses = []
    val_accs = []

    for x_val_batch, y_val_batch in val_dataset:
        val_preds = model(x_val_batch)
        val_loss = loss_fn(y_val_batch, val_preds)

        val_acc = tf.reduce_mean(
            tf.cast(
                tf.argmax(val_preds, axis=1) == tf.argmax(y_val_batch, axis=1),
                tf.float32,
            )
        )
        val_losses.append(val_loss.numpy())
        val_accs.append(val_acc.numpy())

    # Calculate epoch averages
    mean_loss = np.mean(epoch_losses)
    mean_acc = np.mean(epoch_accs) * 100
    mean_val_loss = np.mean(val_losses)
    mean_val_acc = np.mean(val_accs) * 100

    print(
      f"Epoch {epoch + 1:2d}/{epochs} - Loss: {mean_loss:.4f} - Accuracy:"
      f" {mean_acc:.2f}% - Val Loss: {mean_val_loss:.4f} - Val Accuracy:"
      f" {mean_val_acc:.2f}%")


        # --- EARLY STOPPING & WEIGHT RESTORATION ---
    if mean_val_loss < best_val_loss:
        best_val_loss = mean_val_loss
        patience_counter = 0
        best_weights = model.get_weights()  # Saves all W and b arrays in memory
        best_epoch = epoch + 1
    else:
        patience_counter += 1
        if patience_counter > patience:
            model.set_weights(best_weights)  # Restores all W and b arrays
        print(
            f"\nEarly stopping triggered at Epoch {epoch + 1}. Restored best"
            f" weights from Epoch {best_epoch}."
        )
        break

Starting Training...
---------------------------------------------
Epoch  1/100 - Loss: 0.2795 - Accuracy: 91.67% - Val Loss: 0.1452 - Val Accuracy: 95.70%
Epoch  2/100 - Loss: 0.1204 - Accuracy: 96.38% - Val Loss: 0.1215 - Val Accuracy: 96.64%
Epoch  3/100 - Loss: 0.0813 - Accuracy: 97.51% - Val Loss: 0.0841 - Val Accuracy: 97.30%
Epoch  4/100 - Loss: 0.0610 - Accuracy: 98.06% - Val Loss: 0.0684 - Val Accuracy: 97.95%
Epoch  5/100 - Loss: 0.0474 - Accuracy: 98.52% - Val Loss: 0.0685 - Val Accuracy: 97.93%

Early stopping triggered at Epoch 5. Restored best weights from Epoch 4.
